# NACLIP with the VS Code Colab extension

Select **Kernel → Colab → your existing T4 server**. The notebook is currently configured to run **PC59 full validation with guided spatial bias, τ=0.33, temperature=0.10 and PAMR off**. Run the initialization cell, then the **PC59 guided full validation** cell below. The isolated environment and Drive are already prepared on the current runtime. Live iteration output appears below the evaluation cell. An optional ADE20K guided subset pilot is available after the PC59 cell.

Initialization defaults to `SOURCE_MODE='github'` and clones the current pushed source from `https://github.com/Chowkey/NACLIP.git`, branch `main`, when `/content/NACLIP` is absent. Use `SOURCE_MODE='synced'` only after explicitly syncing local source to the selected runtime; that mode verifies files but does not upload them. `UPDATE_SOURCE=True` fetches and fast-forwards a clean checkout after active jobs finish. Existing non-Git folders are preserved in GitHub mode.

Datasets and results persist in Drive. NACLIP is training-free. Optional preparation and upstream evaluation remain disabled.


In [10]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import time

def run_remote(command, **kwargs):
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    kwargs.setdefault('env', environment)
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)

PROJECT = Path('/content/NACLIP')
PYTHON = Path('/content/naclip-env/bin/python')
REPO_URL = 'https://github.com/Chowkey/NACLIP.git'
REPO_BRANCH = 'main'
UPDATE_SOURCE = False  # Enable only after active jobs have finished.

if not Path('/content').is_dir():
    raise RuntimeError('Select a Colab kernel before running these cells.')
# The current source is pushed to REPO_URL/REPO_BRANCH.
# Choose 'synced' only after explicitly syncing source to this runtime.
SOURCE_MODE = 'github'
if SOURCE_MODE == 'synced':
    if not PROJECT.is_dir():
        raise RuntimeError('Sync this local checkout to the selected runtime first: python3 tools/colab_cli.py sync --session naclip')
    EXPECTED_SHA256 = {'naclip.py': '542ecc1c37e50fc26751bce4722c997494f1249f91ccb2b82eef32b2b4977178', 'clip/model.py': 'c4cee9861b75035273527040f6ee4360f7845ee74dee6c2ae25e895cfc3ed585', 'clip/spatial_bias.py': '1f6e2021782e52d5561e8e021e91e0d1118efa3775dad551fd88593b31d3c395', 'tools/run_baseline.py': '65f6f58a3e333e7b6e0c82645898b4106df6cbc388621ae2a7d9bef433cb1292', 'tools/boundary_metric.py': '97b7ec29bbf16f72c4f3a430b24404a22c87fa99fa62284d02c6add6052cd907', 'tools/boundary_metrics.py': '304ea826cac7dfeab8ffe73d64e4539d10d02100a342c54aaf00c60fa3b107d2', 'custom_datasets.py': '9f7dced7ef0c8e89d5545fbb938591fdd478db7088f0617934ce557fc4e853ee', 'configs/base_config.py': '703c656e0fe2161ecabe578d3d29415359f600b22a2b2ae98a3ed9669089eedd', 'configs/cfg_context59.py': '23145571f8b82cd6a5c515333b3ad539b184d86ffdb8457c205ccd4c06625e9a', 'configs/cls_context59.txt': 'cd808465f06a16169edc25580073f0eda347e64f06cc1c2e24acd0b237f6497d'}

    for name, expected in EXPECTED_SHA256.items():
        path = PROJECT / name
        if not path.is_file() or hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest() != expected:
            raise RuntimeError(f'Source mismatch: {name}. Check that the runtime checkout matches the current repository source.')
    print('Using the verified current proposal source already synced to', PROJECT)
elif SOURCE_MODE == 'github':
    source_started = time.perf_counter()
    if not PROJECT.exists():
        run_remote(['git', 'clone', '--depth', '1', '--single-branch',
                    '--branch', REPO_BRANCH, REPO_URL, str(PROJECT)])
    elif not (PROJECT / '.git').is_dir():
        raise RuntimeError(
            f'{PROJECT} already exists without Git metadata (possibly uploaded source). '
            'Choose a new PROJECT path to clone alongside it. Existing files were preserved.')
    else:
        origin = subprocess.check_output(
            ['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
        if origin.removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git').rstrip('/'):
            raise RuntimeError('Existing checkout has a different origin. Choose another PROJECT path.')
        if UPDATE_SOURCE:
            if subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT, text=True).strip():
                raise RuntimeError('Checkout has local changes; preserve them before updating source.')
            branch = subprocess.check_output(
                ['git', 'branch', '--show-current'], cwd=PROJECT, text=True).strip()
            if branch != REPO_BRANCH:
                raise RuntimeError(f'Existing branch is {branch!r}; expected {REPO_BRANCH!r}.')
            run_remote(['git', 'fetch', 'origin', REPO_BRANCH], cwd=PROJECT)
            run_remote(['git', 'merge', '--ff-only', 'FETCH_HEAD'], cwd=PROJECT)
        else:
            print('Keeping existing checkout. Set UPDATE_SOURCE=True to fetch pushed changes.')
    print(f'Source preparation: {time.perf_counter() - source_started:.2f} seconds')
    run_remote(['git', 'log', '-1', '--format=Commit: %H%nSubject: %s'], cwd=PROJECT)
else:
    raise ValueError("SOURCE_MODE must be 'synced' or 'github'.")
required = ['tools/colab_setup.py', 'tools/colab_smoke.py', 'tools/run_baseline.py',
            'tools/prepare_datasets.py', 'tools/colab_eval.py', 'naclip.py',
            'clip/spatial_bias.py', 'tools/boundary_metric.py', 'tools/boundary_metrics.py',
            'tools/proposal_smoke.py']
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise RuntimeError(
        'Repository checkout is missing: ' + ', '.join(missing) +
        '. Commit and push the required local scripts, then update the checkout.')
run_remote(['nvidia-smi'])
print('Remote project:', PROJECT)


Keeping existing checkout. Set UPDATE_SOURCE=True to fetch pushed changes.
Source preparation: 0.01 seconds
Commit: 8671a3dd4613b5bb32e3904af3b93d241c53db36


Subject: Implement guided spatial bias and boundary evaluation
Tue Nov 17 12:53:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |         

## Install isolated dependencies
Run once per new runtime. Downloads can take several minutes. The final import check must succeed before continuing.

In [11]:
run_remote([sys.executable, '-u', str(PROJECT / 'tools/colab_setup.py')], cwd=PROJECT)


Tue Nov 17 12:53:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## GPU inference check
Downloads the pretrained CLIP weights on first use and checks finite output on a synthetic image. This does not measure segmentation quality.

In [12]:
run_remote([str(PYTHON), '-u', 'tools/proposal_smoke.py'], cwd=PROJECT)


11/17 12:53:59 - mmengine - WARNING - The prefix is not set in metric class IoUMetric.
11/17 12:53:59 - mmengine - INFO - per class results:
11/17 12:53:59 - mmengine - INFO - 
+-------+-----+-----+
| Class | IoU | Acc |
+-------+-----+-----+
|   0   | 0.0 | 0.0 |
|   1   | 0.0 | 0.0 |
|   2   | nan | nan |
|   3   | nan | nan |
|   4   | nan | nan |
|   5   | nan | nan |
|   6   | nan | nan |
|   7   | nan | nan |
|   8   | nan | nan |
|   9   | nan | nan |
|   10  | nan | nan |
|   11  | nan | nan |
|   12  | nan | nan |
|   13  | nan | nan |
|   14  | nan | nan |
|   15  | nan | nan |
|   16  | nan | nan |
|   17  | nan | nan |
|   18  | nan | nan |
|   19  | 0.0 | nan |
+-------+-----+-----+
PROPOSAL_SMOKE fixed: passed
11/17 12:53:59 - mmengine - INFO - per class results:
11/17 12:53:59 - mmengine - INFO - 
+-------+-----+-----+
| Class | IoU | Acc |
+-------+-----+-----+
|   0   | 0.0 | 0.0 |
|   1   | 0.0 | 0.0 |
|   2   | nan | nan |
|   3   | nan | nan |
|   4   | nan | nan |


## Optional: mount Drive
Run only if your data/results live on Drive. Complete Google authorization yourself.

In [13]:
MOUNT_DRIVE = True  # Set to False to skip mounting Google Drive.
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


## Optional: upstream dataset evaluation

This is the simpler upstream evaluation entry point. For ADE20K/PC59 baseline runs with saved predictions, timing, and `summary.json`, use the baseline cells below instead. Set paths to an already prepared dataset and a persistent results folder. VOC21 expects JPEGImages, SegmentationClass, and ImageSets/Segmentation/val.txt under DATA_ROOT. Local computer paths are unavailable here.


In [ ]:
RUN_EVALUATION = False  # Set to False to skip evaluation. Set True when ready.
CONFIG = 'configs/cfg_voc21.py'
DATA_ROOT = '/content/drive/MyDrive/datasets/VOCdevkit/VOC2012'
WORK_DIR = '/content/drive/MyDrive/NACLIP-results/voc21'
if RUN_EVALUATION:
    if WORK_DIR.startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Drive first to persist evaluation results.')
    run_remote([str(PYTHON), '-u', 'tools/colab_eval.py', '--config', CONFIG,
                    '--data-root', DATA_ROOT, '--work-dir', WORK_DIR], cwd=PROJECT)
else:
    print('Evaluation disabled. Set dataset paths and RUN_EVALUATION=True when ready.')


## Optional: prepare ADE20K and PC59

Skip this if the datasets are already prepared in Drive. Set `PREPARE_DATASETS=True` only when preparation is needed, after mounting Drive. This follows the proposal dataset notebook and runs on the Colab notebook Python without model dependencies.


In [ ]:
PREPARE_DATASETS = False
if PREPARE_DATASETS:
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Google Drive first.')
    run_remote([sys.executable, '-u', 'tools/prepare_datasets.py',
                '--data-root', '/content/drive/MyDrive/datasets'], cwd=PROJECT)
else:
    print('Dataset preparation disabled; reuse the prepared Drive datasets.')


## PC59 guided full validation

Ready to run: **5,105 images, guided spatial bias, τ=0.33, temperature=0.10, PAMR off**. These parameters were selected before this evaluation from unlabeled ADE20K and PC59 feature distributions.

Run initialization above, then this cell. It streams evaluation output here and saves a console log. Every execution creates a named folder under `MyDrive/NACLIP-results/guided/pc59_guided_full_tau0p33_temp0p10_<UTC timestamp>/`, with a separate Markdown results report beside it. The report updates automatically when the run completes, fails, or is interrupted.

`progress.json` updates every 25 images and on the last image. `summary.json` contains final mIoU, overall/mean class accuracy, pooled 3-pixel boundary and interior accuracy, timing and validity. Configuration, hashes, packages and image IDs are also saved. Full runs do not export prediction panels. Interrupting the cell stops its subprocess; rerunning creates a fresh output folder. Avoid another simultaneous evaluation on this GPU.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json

RUN_BASELINE = True
DATASET = 'pc59'
STAGE = 'full'
CONDITION = 'guided'
TAU = 0.33
TEMPERATURE = 0.10
SEED = 42
DATA_ROOT = Path('/content/drive/MyDrive/datasets/VOCdevkit/VOC2010')
RESULTS_ROOT = Path('/content/drive/MyDrive/NACLIP-results/guided')

if not RUN_BASELINE:
    raise RuntimeError('Evaluation is disabled. Set RUN_BASELINE=True to run.')
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Mount Drive using the mount cell before evaluation.')
if not PYTHON.is_file():
    raise RuntimeError('Run isolated dependency setup first.')
for name in ['JPEGImages', 'SegmentationClassContext', 'ImageSets/SegmentationContext/val.txt']:
    if not (DATA_ROOT / name).exists():
        raise RuntimeError(f'Missing prepared PC59 data: {DATA_ROOT / name}')
ids = (DATA_ROOT / 'ImageSets/SegmentationContext/val.txt').read_text().split()
if len(ids) != 5105:
    raise RuntimeError(f'Expected 5,105 validation IDs, found {len(ids)}.')
EXPECTED_SHA256 = {'naclip.py': '542ecc1c37e50fc26751bce4722c997494f1249f91ccb2b82eef32b2b4977178', 'clip/model.py': 'c4cee9861b75035273527040f6ee4360f7845ee74dee6c2ae25e895cfc3ed585', 'clip/spatial_bias.py': '1f6e2021782e52d5561e8e021e91e0d1118efa3775dad551fd88593b31d3c395', 'tools/run_baseline.py': '65f6f58a3e333e7b6e0c82645898b4106df6cbc388621ae2a7d9bef433cb1292', 'tools/boundary_metric.py': '97b7ec29bbf16f72c4f3a430b24404a22c87fa99fa62284d02c6add6052cd907', 'tools/boundary_metrics.py': '304ea826cac7dfeab8ffe73d64e4539d10d02100a342c54aaf00c60fa3b107d2', 'custom_datasets.py': '9f7dced7ef0c8e89d5545fbb938591fdd478db7088f0617934ce557fc4e853ee', 'configs/base_config.py': '703c656e0fe2161ecabe578d3d29415359f600b22a2b2ae98a3ed9669089eedd', 'configs/cfg_context59.py': '23145571f8b82cd6a5c515333b3ad539b184d86ffdb8457c205ccd4c06625e9a', 'configs/cls_context59.txt': 'cd808465f06a16169edc25580073f0eda347e64f06cc1c2e24acd0b237f6497d'}

for name, expected in EXPECTED_SHA256.items():
    path = PROJECT / name
    if not path.is_file() or hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest() != expected:
        raise RuntimeError(f'Source mismatch: {name}. Check that the runtime checkout matches the current repository source.')
active = []
for path in Path('/proc').glob('[0-9]*/cmdline'):
    try:
        args = path.read_bytes().split(b'\0')
        names = [Path(arg.decode(errors='replace')).name for arg in args if arg]
        if any(name in {'run_baseline.py', 'analyze_guidance.py', 'proposal_smoke.py', 'prepare_datasets.py'} for name in names):
            active.append(path.parent.name)
    except OSError:
        pass
if active:
    raise RuntimeError(f'A project job is already running (PIDs {active}). Wait for it to finish.')

RUN_ID = 'pc59_guided_full_tau0p33_temp0p10_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK_DIR = RESULTS_ROOT / RUN_ID
BASELINE_WORK_DIR = WORK_DIR
REPORT = RESULTS_ROOT / (RUN_ID + '.md')
LOG = RESULTS_ROOT / (RUN_ID + '.log')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
STARTED_UTC = datetime.now(timezone.utc).isoformat()

# Keep the actual work directory empty until the evaluation runner creates it.
if WORK_DIR.exists():
    raise RuntimeError('Output directory already exists. Re-run this cell for a fresh run ID.')

def write_report(status, summary=None, error=None):
    lines = [
        '# PC59 guided full-validation result', '',
        f'- Status: {status}', f'- Started (UTC): {STARTED_UTC}',
        f'- Run: `{RUN_ID}`', '- Dataset: PC59 validation, 5,105 images',
        f'- Condition: guided; tau={TAU}; temperature={TEMPERATURE}; epsilon=1e-6',
        '- PAMR: disabled (0 iterations)', '- Gaussian sigma: 5; reduced architecture; NACLIP key-key attention',
        f'- Seed: {SEED}', f'- Data: `{DATA_ROOT}`', f'- Results: `{WORK_DIR}`',
        f'- Console log: `{LOG}`',
        '- Boundary: both sides of valid horizontal/vertical class transitions, Euclidean distance <=3 pixels, original annotation resolution, ignore=255; pooled pixel accuracy.',
        '- Hyperparameter provenance: selected before this run from unlabeled 32-image ADE20K and PC59 feature distributions; no segmentation-accuracy tuning.', '',
        '## Measurements', '',
    ]
    if summary is None:
        lines += ['Final metrics are pending. Read progress.json for completed/total images.']
    else:
        metrics = summary['metrics']
        lines += ['| Metric | Value |', '| --- | ---: |']
        for label, key in [('mIoU (%)', 'mIoU'), ('Overall pixel accuracy (%)', 'aAcc'),
                           ('Mean class accuracy (%)', 'mAcc'),
                           ('Boundary accuracy (%)', 'localization/boundary_accuracy'),
                           ('Interior accuracy (%)', 'localization/interior_accuracy')]:
            value = metrics.get(key)
            rendered = 'N/A' if value is None else f'{value:.4f}'
            lines.append(f'| {label} | {rendered} |')
        lines += [f"| Images | {summary['images']} |",
                  f"| Evaluation wall time (seconds) | {summary['test_wall_seconds']:.2f} |",
                  f"| Peak GPU allocation (GiB) | {summary['peak_gpu_allocated_gib']:.3f} |", '',
                  f"- baseline_valid: {summary['baseline_valid']}",
                  f"- PAMR failures: {len(summary['pamr_failures'])}",
                  '- Exact configuration, source hashes and packages are saved in config.py, manifest.json and packages.txt.']
    if error:
        lines += ['', '## Execution issue', '', str(error)]
    lines += ['', '## Comparison', '',
              'A matched full PC59 fixed-bias run with PAMR off is required to measure the proposal improvement. The historical PC59 score of 38.36 mIoU used PAMR on and is not a controlled baseline for this run.', '']
    REPORT.write_text('\n'.join(lines), encoding='utf-8')

command = [str(PYTHON), '-u', 'tools/run_baseline.py',
           '--dataset', DATASET, '--stage', STAGE, '--condition', CONDITION,
           '--guidance-tau', str(TAU), '--guidance-temperature', str(TEMPERATURE),
           '--guidance-epsilon', '1e-6', '--seed', str(SEED),
           '--data-root', str(DATA_ROOT), '--work-dir', str(WORK_DIR)]
print('Results:', WORK_DIR, flush=True)
print('Markdown report:', REPORT, flush=True)
print('Log:', LOG, flush=True)
write_report('running')
environment = os.environ.copy()
environment['MPLBACKEND'] = 'Agg'
process = None
try:
    with LOG.open('w', encoding='utf-8') as logfile:
        process = subprocess.Popen(command, cwd=PROJECT, env=environment,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            logfile.write(line)
            logfile.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    summary = json.loads((WORK_DIR / 'summary.json').read_text())
    if summary['images'] != 5105 or not summary['baseline_valid'] or summary['pamr_failures']:
        raise RuntimeError('The full run did not pass completion checks. Inspect summary.json.')
    if (summary['condition'] != 'guided' or summary['resolved_model']['pamr_steps'] != 0
            or summary['resolved_model']['guidance_tau'] != TAU
            or summary['resolved_model']['guidance_temperature'] != TEMPERATURE):
        raise RuntimeError('Resolved model settings differ from the requested guided run.')
    write_report('completed', summary=summary)
    print('Completed; final metrics and report are saved to Drive.', flush=True)
except BaseException as error:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    write_report('interrupted' if isinstance(error, KeyboardInterrupt) else 'failed', error=repr(error))
    raise


## ADE20K guided validation (subset pilot)

Ready to run: **32-image ADE20K subset by default, guided spatial bias, tau=0.33, temperature=0.10, PAMR off**. This reuses the same tau and temperature as PC59. After reviewing the subset, change `STAGE` to `full` to evaluate all 2,000 validation images.

Run initialization above, mount Drive, and make sure no other evaluation is active on this GPU. Each execution creates a named folder under `MyDrive/NACLIP-results/guided/ade20k_guided_<stage>_tau0p33_temp0p10_<UTC timestamp>/`, with a Markdown report and console log beside it. The code checks the ADE20K source hashes and dataset layout before launching.

The subset is a pipeline check, not a benchmark result. Full evaluation reports mIoU, overall/mean class accuracy, pooled 3-pixel boundary and interior accuracy, timing, and validity.

In [14]:
from pathlib import Path

data = Path('/content/drive/MyDrive/datasets/ADEChallengeData2016')
print((data / 'images/validation').is_dir())
print((data / 'annotations/validation').is_dir())

True
True


In [15]:
from datetime import datetime, timezone
import hashlib
import json

RUN_BASELINE = True
DATASET = 'ade20k'
STAGE = 'full'
SUBSET_SIZE = 32
CONDITION = 'guided'
TAU = 0.33
TEMPERATURE = 0.10
SEED = 42
DATA_ROOT = Path('/content/drive/MyDrive/datasets/ADEChallengeData2016')
RESULTS_ROOT = Path('/content/drive/MyDrive/NACLIP-results/guided')

if not RUN_BASELINE:
    raise RuntimeError('Evaluation is disabled. Set RUN_BASELINE=True to run.')
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Mount Drive using the mount cell before evaluation.')
if not PYTHON.is_file():
    raise RuntimeError('Run isolated dependency setup first.')
for name in ['images/validation', 'annotations/validation']:
    if not (DATA_ROOT / name).is_dir():
        raise RuntimeError(f'Missing prepared ADE20K data: {DATA_ROOT / name}')
image_count = len(list((DATA_ROOT / 'images/validation').glob('*.jpg')))
annotation_count = len(list((DATA_ROOT / 'annotations/validation').glob('*.png')))
if image_count != 2000 or annotation_count != 2000:
    raise RuntimeError(f'Expected 2,000 ADE20K validation images and masks; found {image_count} images and {annotation_count} masks.')
EXPECTED_SHA256 = {'naclip.py': '542ecc1c37e50fc26751bce4722c997494f1249f91ccb2b82eef32b2b4977178', 'clip/model.py': 'c4cee9861b75035273527040f6ee4360f7845ee74dee6c2ae25e895cfc3ed585', 'clip/spatial_bias.py': '1f6e2021782e52d5561e8e021e91e0d1118efa3775dad551fd88593b31d3c395', 'tools/run_baseline.py': '65f6f58a3e333e7b6e0c82645898b4106df6cbc388621ae2a7d9bef433cb1292', 'tools/boundary_metric.py': '97b7ec29bbf16f72c4f3a430b24404a22c87fa99fa62284d02c6add6052cd907', 'tools/boundary_metrics.py': '304ea826cac7dfeab8ffe73d64e4539d10d02100a342c54aaf00c60fa3b107d2', 'custom_datasets.py': '9f7dced7ef0c8e89d5545fbb938591fdd478db7088f0617934ce557fc4e853ee', 'configs/base_config.py': '703c656e0fe2161ecabe578d3d29415359f600b22a2b2ae98a3ed9669089eedd', 'configs/cfg_ade20k.py': '8f941b77d5e06482da429baf293aa1b5ae869cee514443492df1d31b2a2ad670', 'configs/cls_ade20k.txt': 'e53ded02c488bdb903ef000a2d207e4ac4f2bb4502a72572e4aab7c2504aef3a'}

for name, expected in EXPECTED_SHA256.items():
    path = PROJECT / name
    if not path.is_file() or hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest() != expected:
        raise RuntimeError(f'Source mismatch: {name}. Check that the runtime checkout matches the current repository source.')
active = []
for path in Path('/proc').glob('[0-9]*/cmdline'):
    try:
        args = path.read_bytes().split(b'\0')
        names = [Path(arg.decode(errors='replace')).name for arg in args if arg]
        if any(name in {'run_baseline.py', 'analyze_guidance.py', 'proposal_smoke.py', 'prepare_datasets.py'} for name in names):
            active.append(path.parent.name)
    except OSError:
        pass
if active:
    raise RuntimeError(f'A project job is already running (PIDs {active}). Wait for it to finish.')

TARGET_IMAGES = min(SUBSET_SIZE, 2000) if STAGE == 'subset' else 2000
RUN_ID = f'ade20k_guided_{STAGE}_tau0p33_temp0p10_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK_DIR = RESULTS_ROOT / RUN_ID
BASELINE_WORK_DIR = WORK_DIR
REPORT = RESULTS_ROOT / (RUN_ID + '.md')
LOG = RESULTS_ROOT / (RUN_ID + '.log')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
STARTED_UTC = datetime.now(timezone.utc).isoformat()

# Keep the actual work directory empty until the evaluation runner creates it.
if WORK_DIR.exists():
    raise RuntimeError('Output directory already exists. Re-run this cell for a fresh run ID.')

def write_report(status, summary=None, error=None):
    lines = [
        f'# ADE20K guided {STAGE}-validation result', '',
        f'- Status: {status}', f'- Started (UTC): {STARTED_UTC}',
        f'- Run: `{RUN_ID}`', f'- Dataset: ADE20K validation, {TARGET_IMAGES} images ({STAGE})',
        f'- Condition: guided; tau={TAU}; temperature={TEMPERATURE}; epsilon=1e-6',
        '- PAMR: disabled (0 iterations)', '- Gaussian sigma: 5; reduced architecture; NACLIP key-key attention',
        f'- Seed: {SEED}', f'- Data: `{DATA_ROOT}`', f'- Results: `{WORK_DIR}`',
        f'- Console log: `{LOG}`',
        '- Boundary: both sides of valid horizontal/vertical class transitions, Euclidean distance <=3 pixels, original annotation resolution, ignore=255; pooled pixel accuracy.',
        '- Hyperparameter provenance: selected before this run from unlabeled 32-image ADE20K and PC59 feature distributions; no segmentation-accuracy tuning.', '',
        '## Measurements', '',
    ]
    if summary is None:
        lines += ['Final metrics are pending. Read progress.json for completed/total images.']
    else:
        metrics = summary['metrics']
        lines += ['| Metric | Value |', '| --- | ---: |']
        for label, key in [('mIoU (%)', 'mIoU'), ('Overall pixel accuracy (%)', 'aAcc'),
                           ('Mean class accuracy (%)', 'mAcc'),
                           ('Boundary accuracy (%)', 'localization/boundary_accuracy'),
                           ('Interior accuracy (%)', 'localization/interior_accuracy')]:
            value = metrics.get(key)
            rendered = 'N/A' if value is None else f'{value:.4f}'
            lines.append(f'| {label} | {rendered} |')
        lines += [f"| Images | {summary['images']} |",
                  f"| Evaluation wall time (seconds) | {summary['test_wall_seconds']:.2f} |",
                  f"| Peak GPU allocation (GiB) | {summary['peak_gpu_allocated_gib']:.3f} |", '',
                  f"- baseline_valid: {summary['baseline_valid']}",
                  f"- PAMR failures: {len(summary['pamr_failures'])}",
                  '- Exact configuration, source hashes and packages are saved in config.py, manifest.json and packages.txt.']
    if error:
        lines += ['', '## Execution issue', '', str(error)]
    lines += ['', '## Comparison', '',
              'Compare against a matched ADE20K fixed-bias run with PAMR off using the same stage, seed, and image IDs. Historical PAMR-on ADE20K scores are not a controlled comparison for this run.', '']
    REPORT.write_text('\n'.join(lines), encoding='utf-8')

command = [str(PYTHON), '-u', 'tools/run_baseline.py',
           '--dataset', DATASET, '--stage', STAGE, '--condition', CONDITION,
           '--guidance-tau', str(TAU), '--guidance-temperature', str(TEMPERATURE),
           '--guidance-epsilon', '1e-6', '--seed', str(SEED),
           '--data-root', str(DATA_ROOT), '--work-dir', str(WORK_DIR),
           '--subset-size', str(SUBSET_SIZE)]
print('Results:', WORK_DIR, flush=True)
print('Markdown report:', REPORT, flush=True)
print('Log:', LOG, flush=True)
write_report('running')
environment = os.environ.copy()
environment['MPLBACKEND'] = 'Agg'
process = None
try:
    with LOG.open('w', encoding='utf-8') as logfile:
        process = subprocess.Popen(command, cwd=PROJECT, env=environment,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            logfile.write(line)
            logfile.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    summary = json.loads((WORK_DIR / 'summary.json').read_text())
    if summary['images'] != TARGET_IMAGES or not summary['baseline_valid'] or summary['pamr_failures']:
        raise RuntimeError('The ADE20K run did not pass completion checks. Inspect summary.json.')
    if (summary['condition'] != 'guided' or summary['resolved_model']['pamr_steps'] != 0
            or summary['resolved_model']['guidance_tau'] != TAU
            or summary['resolved_model']['guidance_temperature'] != TEMPERATURE):
        raise RuntimeError('Resolved model settings differ from the requested guided run.')
    write_report('completed', summary=summary)
    print('Completed; final metrics and report are saved to Drive.', flush=True)
except BaseException as error:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    write_report('interrupted' if isinstance(error, KeyboardInterrupt) else 'failed', error=repr(error))
    raise


Results: /content/drive/MyDrive/NACLIP-results/guided/ade20k_guided_full_tau0p33_temp0p10_20261117T125415227085Z
Markdown report: /content/drive/MyDrive/NACLIP-results/guided/ade20k_guided_full_tau0p33_temp0p10_20261117T125415227085Z.md
Log: /content/drive/MyDrive/NACLIP-results/guided/ade20k_guided_full_tau0p33_temp0p10_20261117T125415227085Z.log
/content/naclip-env/lib/python3.10/site-packages/mmseg/datasets/transforms/loading.py:83: UserWarning: `reduce_zero_label` will be deprecated, if you would like to ignore the zero label, please set `reduce_zero_label=True` when dataset initialized
  warnings.warn('`reduce_zero_label` will be deprecated, '
11/17 12:54:19 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.22 (main, Oct  9 2026, 15:21:30) [Clang 22.1.3 ]
    CUDA available: True
    numpy_random_seed: 42
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, 

## Inspect progress and results

Run this after evaluation. While the evaluation cell is busy, view `progress.json` or the adjacent `.log` through the extension Contents view. Final metrics are written to `summary.json` and the separate Markdown report only after evaluation completes. Historical PAMR-on PC59 results are not the controlled fixed/PAMR-off comparison for this guided run.


In [ ]:
from IPython.display import Markdown, display

for name in ['progress.json', 'summary.json']:
    path = WORK_DIR / name
    if path.is_file():
        print(name)
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print(name, 'is not available yet.')
if REPORT.is_file():
    display(Markdown(REPORT.read_text()))


Fixed Gaussian with PAMR OFF, baseline for PC59

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK_DIR = Path('/content/drive/MyDrive/NACLIP-results/fixed') / (
    f'pc59_fixed_full_pamr_off_{stamp}'
)

print('Saving results to:', WORK_DIR, flush=True)

run_remote([
    str(PYTHON), '-u', 'tools/run_baseline.py',
    '--dataset', 'pc59',
    '--stage', 'full',
    '--condition', 'fixed',
    '--seed', '42',
    '--data-root', '/content/drive/MyDrive/datasets/VOCdevkit/VOC2010',
    '--work-dir', str(WORK_DIR),
], cwd=PROJECT)

summary = json.loads((WORK_DIR / 'summary.json').read_text())
REPORT = WORK_DIR / 'results.md'
REPORT.write_text(
    '# PC59 fixed Gaussian — PAMR off\n\n'
    + '```json\n' + json.dumps(summary, indent=2) + '\n```\n'
)
print('Markdown report:', REPORT)

## ADE20K overlap calibration paired pilot

Runs the same fixed 32-image ADE20K subset (seed 42) twice with guided spatial bias and PAMR off: standard sliding-window merge, then overlap-disagreement calibration. It checks matching image IDs and writes both summaries plus a paired comparison to Drive. This is a development pilot, not a benchmark claim. Run after initialization, dependency setup, Drive mount, and after syncing the current local source.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import subprocess

RUN_PAIRED_PILOT = True
DATA_ROOT = Path('/content/drive/MyDrive/datasets/ADEChallengeData2016')
RESULTS_ROOT = Path('/content/drive/MyDrive/NACLIP-results/window-calibration')
SEED = 42
SUBSET_SIZE = 32
TAU = 0.33
TEMPERATURE = 0.10
EXPECTED = {
    'naclip.py': '542ecc1c37e50fc26751bce4722c997494f1249f91ccb2b82eef32b2b4977178',
    'tools/run_baseline.py': '65f6f58a3e333e7b6e0c82645898b4106df6cbc388621ae2a7d9bef433cb1292',
}
if not RUN_PAIRED_PILOT:
    raise RuntimeError('Pilot is disabled. Set RUN_PAIRED_PILOT=True to run.')
if not PYTHON.is_file():
    raise RuntimeError('Run isolated dependency setup first.')
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Mount Drive using the mount cell before evaluation.')
for relative, expected in EXPECTED.items():
    path = PROJECT / relative
    actual = hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Source mismatch: {relative}. Sync the current local proposal source before running.')
for relative in ['images/validation', 'annotations/validation']:
    if not (DATA_ROOT / relative).is_dir():
        raise RuntimeError(f'Missing ADE20K data: {DATA_ROOT / relative}')
active = []
for path in Path('/proc').glob('[0-9]*/cmdline'):
    try:
        names = [Path(arg.decode(errors='replace')).name for arg in path.read_bytes().split(b'\0') if arg]
        if 'run_baseline.py' in names:
            active.append(path.parent.name)
    except OSError:
        pass
if active:
    raise RuntimeError(f'A project evaluation is already running (PIDs {active}). Wait for it to finish.')

RUN_ID = 'ade20k_paired_seed42_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_ROOT = RESULTS_ROOT / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
common = [str(PYTHON), '-u', str(PROJECT / 'tools/run_baseline.py'),
          '--dataset', 'ade20k', '--stage', 'subset', '--subset-size', str(SUBSET_SIZE),
          '--seed', str(SEED), '--condition', 'guided', '--guidance-tau', str(TAU),
          '--guidance-temperature', str(TEMPERATURE), '--data-root', str(DATA_ROOT)]
for label, mode in [('standard', 'off'), ('calibrated', 'disagreement')]:
    work_dir = RUN_ROOT / label
    command = common + ['--work-dir', str(work_dir), '--window-calibration', mode,
                        '--window-calibration-strength', '0.5', '--window-confidence-power', '1.0']
    print('Running', label, flush=True)
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'{label} runner failed with exit status {process.returncode}')

standard = json.loads((RUN_ROOT / 'standard/summary.json').read_text())
calibrated = json.loads((RUN_ROOT / 'calibrated/summary.json').read_text())
standard_ids = json.loads((RUN_ROOT / 'standard/image_ids.json').read_text())
calibrated_ids = json.loads((RUN_ROOT / 'calibrated/image_ids.json').read_text())
if standard_ids != calibrated_ids:
    raise RuntimeError('Paired pilot image IDs differ; comparison is invalid.')
if standard['images'] != SUBSET_SIZE or calibrated['images'] != SUBSET_SIZE:
    raise RuntimeError('One paired run did not evaluate all requested images.')
comparison = {
    'run_id': RUN_ID, 'dataset': 'ade20k', 'stage': 'subset', 'seed': SEED,
    'image_count': len(standard_ids), 'image_ids_match': True,
    'condition': 'guided, tau=0.33, temperature=0.10, PAMR off',
    'calibration': {'strength': 0.5, 'confidence_power': 1.0},
    'standard': standard, 'calibrated': calibrated,
    'delta_calibrated_minus_standard': {
        key: calibrated['metrics'].get(key) - standard['metrics'].get(key)
        for key in ['mIoU', 'aAcc', 'mAcc', 'localization/boundary_accuracy',
                    'localization/interior_accuracy']
        if calibrated['metrics'].get(key) is not None and standard['metrics'].get(key) is not None
    },
}
(RUN_ROOT / 'paired_comparison.json').write_text(json.dumps(comparison, indent=2))
print('PAIRED_PILOT ' + json.dumps({
    'run_id': RUN_ID, 'image_ids_match': True,
    'standard_metrics': standard['metrics'], 'calibrated_metrics': calibrated['metrics'],
    'delta': comparison['delta_calibrated_minus_standard'],
    'standard_seconds': standard['test_wall_seconds'],
    'calibrated_seconds': calibrated['test_wall_seconds'],
    'standard_valid': standard['baseline_valid'], 'calibrated_valid': calibrated['baseline_valid'],
    'results': str(RUN_ROOT),
}, sort_keys=True), flush=True)
